<a href="https://colab.research.google.com/github/raki-rankawat/thesis-v1/blob/master/Model_KD_Combined.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Model_KD_Combined

**Knowledge Distillation — 2 Teachers × 2 Students × 2 Init Strategies = 8 Runs**

| Role | Model | Checkpoint |
|---|---|---|
| Teacher A | `VGG_Pretrained` | `vgg_pretrained_seed_85.pth` |
| Teacher B | `ResNet_Pretrained` | `resnet_pretrained_seed_52.pth` |
| Student 1 | `VWW_MobileNetV2` | `mobilenetv2_seed_74.pth` |
| Student 2 | `VWW_MobileNetV3` | `mobilenetv3_seed_74.pth` |

### Re-run behaviour
Every run checks its output checkpoint before training:
- **Checkpoint exists + accuracy ≥ threshold** → skipped automatically
- **Checkpoint exists + accuracy < threshold** → re-trains and overwrites
- **No checkpoint** → trains from scratch

Set `FORCE_RETRAIN = True` in the Config cell to ignore checkpoints and retrain everything.

**Prerequisites:** `Model_VGG_Pretrained`, `Model_ResNet_Pretrained`, `Model_MobileNetV2`, `Model_MobileNetV3`

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import sys, shutil, os
UTILS_SRC = "/content/drive/My Drive/stm32-thesis/utils"
if os.path.exists(UTILS_SRC):
    shutil.copytree(UTILS_SRC, "/content/utils", dirs_exist_ok=True)
    sys.path.insert(0, "/content")
    print("✅ utils loaded from Drive")
else:
    sys.path.insert(0, "/content")
    print("⚠️  Place utils/ at: My Drive/stm32-thesis/utils/")

Mounted at /content/drive
✅ utils loaded from Drive


In [2]:
import os, json
import torch

from utils.dataset import prepare_dataset, get_loaders
from utils.models  import (
    VGG_Pretrained, ResNet_Pretrained,
    VWW_MobileNetV2, VWW_MobileNetV3,
)
from utils.train import setup_device, evaluate, train_kd

device = setup_device(seed=41)

Device: cuda


In [3]:
prepare_dataset()
train_loader, val_loader = get_loaders(batch_size=64, augmentation="standard")

1/4 Download
⬇️  Downloading VWW archive...
✅ Download complete: /content/vww_work/vw_coco2014_96.tar.gz
2/4 Extract
📦 Extracting VWW archive...
✅ Extraction complete: /content/vww_work/extracted
3/4 Find root
   Root: /content/vww_work/extracted/vw_coco2014_96
4/4 Manifests
✅ Manifests already exist: /content/drive/My Drive/vww_fixed_split_manifests
Train: 7000 | Val: 1500 | Batch: 64


---
## ⚙️ Config — edit here before each run

In [4]:
# ── Paths ────────────────────────────────────────────────────────────
SAVE_DIR = "/content/drive/My Drive/stm32-thesis/checkpoints"

# Teacher checkpoints — update seed suffix if needed
TEACHER_VGG_CKPT    = f"{SAVE_DIR}/vgg_pretrained_seed_85.pth"
TEACHER_RESNET_CKPT = f"{SAVE_DIR}/resnet_pretrained_seed_85.pth"  # update seed

# Student warm-start checkpoints
MV2_CKPT = f"{SAVE_DIR}/mobilenetv2_seed_74.pth"
MV3_CKPT = f"{SAVE_DIR}/mobilenetv3_seed_74.pth"

# ── KD hyperparameters ────────────────────────────────────────────────
KD_TEMPERATURE = 4.0
KD_ALPHA       = 0.7
KD_EPOCHS      = 60    # was 30 — scratch needs more room to converge
KD_LR_FT      = 3e-4  # fine-tune: lower to preserve warm-start weights
KD_LR_SCR     = 1e-3  # scratch: standard
KD_WD         = 1e-4
KD_PATIENCE   = 15    # was 10 — scratch val is noisy on 1500 samples

# ── Re-run control ────────────────────────────────────────────────────
# Minimum val accuracy to consider a checkpoint 'satisfactory'.
# If existing checkpoint is below this, it will be re-trained.
THRESHOLDS = {
    "vgg_mv2_ft"      : 0.800,
    "vgg_mv2_scratch" : 0.780,
    "vgg_mv3_ft"      : 0.800,
    "vgg_mv3_scratch" : 0.780,
    "resnet_mv2_ft"   : 0.800,
    "resnet_mv2_scratch": 0.780,
    "resnet_mv3_ft"   : 0.800,
    "resnet_mv3_scratch": 0.780,
}

# Set True to retrain everything regardless of existing checkpoints
FORCE_RETRAIN = False

---
## Helper — skip / run logic

In [5]:
def should_run(run_key, ckpt_path, model_cls):
    """
    Returns (run: bool, reason: str, existing_acc: float | None)

    Skips the run if:
      - FORCE_RETRAIN is False, AND
      - checkpoint exists, AND
      - its val accuracy >= THRESHOLDS[run_key]
    """
    if FORCE_RETRAIN:
        return True, "FORCE_RETRAIN=True", None
    if not os.path.exists(ckpt_path):
        return True, "no checkpoint found", None
    # Load and evaluate existing checkpoint
    m = model_cls().to(device)
    m.load_state_dict(torch.load(ckpt_path, map_location=device))
    acc = evaluate(m, val_loader, device)
    del m
    threshold = THRESHOLDS[run_key]
    if acc >= threshold:
        return False, f"acc {acc*100:.2f}% >= threshold {threshold*100:.1f}%", acc
    else:
        return True, f"acc {acc*100:.2f}% < threshold {threshold*100:.1f}% — retraining", acc


# Results registry — populated as runs complete or are skipped
results = {}

def register(run_key, acc, source="trained"):
    results[run_key] = {"val_acc": acc, "source": source}
    print(f"  → Registered [{run_key}]: {acc*100:.2f}%  ({source})")

---
## Load Teachers

In [6]:
# ── Load and verify both teachers ───────────────────────────────────
teacher_vgg = VGG_Pretrained().to(device)
teacher_vgg.load_state_dict(torch.load(TEACHER_VGG_CKPT, map_location=device))
teacher_vgg.eval()
for p in teacher_vgg.parameters(): p.requires_grad = False
vgg_acc = evaluate(teacher_vgg, val_loader, device)
print(f"Teacher VGG_Pretrained    : {vgg_acc*100:.2f}%")

teacher_resnet = ResNet_Pretrained().to(device)
teacher_resnet.load_state_dict(torch.load(TEACHER_RESNET_CKPT, map_location=device))
teacher_resnet.eval()
for p in teacher_resnet.parameters(): p.requires_grad = False
resnet_acc = evaluate(teacher_resnet, val_loader, device)
print(f"Teacher ResNet_Pretrained : {resnet_acc*100:.2f}%")

# Baseline student accuracies (for delta reporting later)
_mv2_tmp = VWW_MobileNetV2().to(device)
_mv2_tmp.load_state_dict(torch.load(MV2_CKPT, map_location=device))
baseline_mv2 = evaluate(_mv2_tmp, val_loader, device); del _mv2_tmp

_mv3_tmp = VWW_MobileNetV3().to(device)
_mv3_tmp.load_state_dict(torch.load(MV3_CKPT, map_location=device))
baseline_mv3 = evaluate(_mv3_tmp, val_loader, device); del _mv3_tmp

print(f"\nStudent MobileNetV2 baseline: {baseline_mv2*100:.2f}%")
print(f"Student MobileNetV3 baseline: {baseline_mv3*100:.2f}%")

for teacher_name, t_acc, baseline in [
    ("VGG", vgg_acc, baseline_mv2), ("VGG", vgg_acc, baseline_mv3),
    ("ResNet", resnet_acc, baseline_mv2), ("ResNet", resnet_acc, baseline_mv3),
]:
    if t_acc <= baseline:
        print(f"⚠️  {teacher_name} teacher ({t_acc*100:.2f}%) is NOT stronger than student ({baseline*100:.2f}%) — KD may not help")

Downloading: "https://download.pytorch.org/models/vgg16_bn-6c64b313.pth" to /root/.cache/torch/hub/checkpoints/vgg16_bn-6c64b313.pth


100%|██████████| 528M/528M [00:06<00:00, 90.4MB/s]


Teacher VGG_Pretrained    : 89.07%
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 106MB/s]


Teacher ResNet_Pretrained : 87.07%

Student MobileNetV2 baseline: 78.40%
Student MobileNetV3 baseline: 79.13%


---
## Teacher: VGG_Pretrained  ·  Student: MobileNetV2


In [7]:
# ── VGG_Pretrained → MobileNetV2 · KD Fine-Tune ────────────────────────
_ckpt = f"{SAVE_DIR}/mv2_kd_vgg_ft.pth"
_run, _reason, _existing_acc = should_run("vgg_mv2_ft", _ckpt, VWW_MobileNetV2)

if not _run:
    print(f"⏭️  Skipping vgg_mv2_ft — {_reason}")
    register("vgg_mv2_ft", _existing_acc, source="existing checkpoint")
else:
    print(f"▶️  Running vgg_mv2_ft — {_reason}")
    _student = VWW_MobileNetV2().to(device)
    _student.load_state_dict(torch.load(MV2_CKPT, map_location=device))
    print(f"MobileNetV2 warm-start baseline: {baseline_mv2*100:.2f}%")
    _acc, _t = train_kd(
        student      = _student,
        teacher      = teacher_vgg,
        train_loader = train_loader,
        val_loader   = val_loader,
        device       = device,
        epochs       = KD_EPOCHS,
        lr           = KD_LR_FT,
        weight_decay = KD_WD,
        temperature  = KD_TEMPERATURE,
        alpha        = KD_ALPHA,
        patience     = KD_PATIENCE,
        save_path    = _ckpt,
    )
    register("vgg_mv2_ft", _acc)

▶️  Running vgg_mv2_ft — no checkpoint found
MobileNetV2 warm-start baseline: 78.40%
Initial student accuracy: 78.40%
Epoch   1/60 | Train 81.23% | Val 78.60% ✅
Epoch   2/60 | Train 81.10% | Val 78.67% ✅
Epoch   3/60 | Train 80.93% | Val 76.33%
Epoch   4/60 | Train 81.49% | Val 77.53%
Epoch   5/60 | Train 82.17% | Val 77.67%
Epoch   6/60 | Train 81.51% | Val 79.20% ✅
Epoch   7/60 | Train 81.00% | Val 78.80%
Epoch   8/60 | Train 82.64% | Val 79.13%
Epoch   9/60 | Train 81.53% | Val 77.93%
Epoch  10/60 | Train 82.03% | Val 80.00% ✅
Epoch  11/60 | Train 81.89% | Val 78.67%
Epoch  12/60 | Train 82.49% | Val 78.00%
Epoch  13/60 | Train 81.96% | Val 78.47%
Epoch  14/60 | Train 81.94% | Val 78.13%
Epoch  15/60 | Train 82.19% | Val 78.00%
Epoch  16/60 | Train 82.87% | Val 78.93%
Epoch  17/60 | Train 82.31% | Val 78.33%
Epoch  18/60 | Train 82.79% | Val 78.60%
Epoch  19/60 | Train 82.94% | Val 78.40%
Epoch  20/60 | Train 82.89% | Val 79.27%
Epoch  21/60 | Train 83.54% | Val 77.87%
Epoch  22/60 

In [8]:
# ── VGG_Pretrained → MobileNetV2 · KD Scratch ──────────────────────────
_ckpt = f"{SAVE_DIR}/mv2_kd_vgg_scratch.pth"
_run, _reason, _existing_acc = should_run("vgg_mv2_scratch", _ckpt, VWW_MobileNetV2)

if not _run:
    print(f"⏭️  Skipping vgg_mv2_scratch — {_reason}")
    register("vgg_mv2_scratch", _existing_acc, source="existing checkpoint")
else:
    print(f"▶️  Running vgg_mv2_scratch — {_reason}")
    _student = VWW_MobileNetV2().to(device)
    print(f"MobileNetV2 random init baseline: random weights, ~50%")
    _acc, _t = train_kd(
        student      = _student,
        teacher      = teacher_vgg,
        train_loader = train_loader,
        val_loader   = val_loader,
        device       = device,
        epochs       = KD_EPOCHS,
        lr           = KD_LR_SCR,
        weight_decay = KD_WD,
        temperature  = KD_TEMPERATURE,
        alpha        = KD_ALPHA,
        patience     = KD_PATIENCE,
        save_path    = _ckpt,
    )
    register("vgg_mv2_scratch", _acc)

▶️  Running vgg_mv2_scratch — no checkpoint found
MobileNetV2 random init baseline: random weights, ~50%
Initial student accuracy: 50.00%
Epoch   1/60 | Train 60.67% | Val 64.40% ✅
Epoch   2/60 | Train 64.97% | Val 67.80% ✅
Epoch   3/60 | Train 68.23% | Val 68.87% ✅
Epoch   4/60 | Train 69.71% | Val 70.07% ✅
Epoch   5/60 | Train 71.07% | Val 71.80% ✅
Epoch   6/60 | Train 71.94% | Val 72.13% ✅
Epoch   7/60 | Train 73.09% | Val 73.07% ✅
Epoch   8/60 | Train 73.17% | Val 73.27% ✅
Epoch   9/60 | Train 74.59% | Val 73.27%
Epoch  10/60 | Train 74.16% | Val 72.40%
Epoch  11/60 | Train 74.51% | Val 73.60% ✅
Epoch  12/60 | Train 75.86% | Val 72.80%
Epoch  13/60 | Train 75.14% | Val 75.67% ✅
Epoch  14/60 | Train 75.84% | Val 75.47%
Epoch  15/60 | Train 75.96% | Val 76.93% ✅
Epoch  16/60 | Train 76.87% | Val 76.27%
Epoch  17/60 | Train 76.51% | Val 76.67%
Epoch  18/60 | Train 76.53% | Val 70.80%
Epoch  19/60 | Train 77.34% | Val 77.87% ✅
Epoch  20/60 | Train 77.76% | Val 75.80%
Epoch  21/60 | Tra

---
## Teacher: VGG_Pretrained  ·  Student: MobileNetV3


In [9]:
# ── VGG_Pretrained → MobileNetV3 · KD Fine-Tune ────────────────────────
_ckpt = f"{SAVE_DIR}/mv3_kd_vgg_ft.pth"
_run, _reason, _existing_acc = should_run("vgg_mv3_ft", _ckpt, VWW_MobileNetV3)

if not _run:
    print(f"⏭️  Skipping vgg_mv3_ft — {_reason}")
    register("vgg_mv3_ft", _existing_acc, source="existing checkpoint")
else:
    print(f"▶️  Running vgg_mv3_ft — {_reason}")
    _student = VWW_MobileNetV3().to(device)
    _student.load_state_dict(torch.load(MV3_CKPT, map_location=device))
    print(f"MobileNetV3 warm-start baseline: {baseline_mv3*100:.2f}%")
    _acc, _t = train_kd(
        student      = _student,
        teacher      = teacher_vgg,
        train_loader = train_loader,
        val_loader   = val_loader,
        device       = device,
        epochs       = KD_EPOCHS,
        lr           = KD_LR_FT,
        weight_decay = KD_WD,
        temperature  = KD_TEMPERATURE,
        alpha        = KD_ALPHA,
        patience     = KD_PATIENCE,
        save_path    = _ckpt,
    )
    register("vgg_mv3_ft", _acc)

▶️  Running vgg_mv3_ft — no checkpoint found
MobileNetV3 warm-start baseline: 79.13%
Initial student accuracy: 79.13%
Epoch   1/60 | Train 81.59% | Val 77.47%
Epoch   2/60 | Train 81.50% | Val 76.87%
Epoch   3/60 | Train 81.96% | Val 77.40%
Epoch   4/60 | Train 82.37% | Val 77.93%
Epoch   5/60 | Train 82.41% | Val 77.87%
Epoch   6/60 | Train 82.43% | Val 76.53%
Epoch   7/60 | Train 82.26% | Val 77.13%
Epoch   8/60 | Train 82.33% | Val 77.80%
Epoch   9/60 | Train 82.43% | Val 77.73%
Epoch  10/60 | Train 82.63% | Val 77.60%
Epoch  11/60 | Train 82.33% | Val 77.60%
Epoch  12/60 | Train 82.90% | Val 77.00%
Epoch  13/60 | Train 83.09% | Val 77.33%
Epoch  14/60 | Train 82.83% | Val 77.27%
Epoch  15/60 | Train 83.74% | Val 77.47%
🛑 Early stopping at epoch 15

✅ Best KD val acc: 79.13%  (3.9 min)
  → Registered [vgg_mv3_ft]: 79.13%  (trained)


In [10]:
# ── VGG_Pretrained → MobileNetV3 · KD Scratch ──────────────────────────
_ckpt = f"{SAVE_DIR}/mv3_kd_vgg_scratch.pth"
_run, _reason, _existing_acc = should_run("vgg_mv3_scratch", _ckpt, VWW_MobileNetV3)

if not _run:
    print(f"⏭️  Skipping vgg_mv3_scratch — {_reason}")
    register("vgg_mv3_scratch", _existing_acc, source="existing checkpoint")
else:
    print(f"▶️  Running vgg_mv3_scratch — {_reason}")
    _student = VWW_MobileNetV3().to(device)
    print(f"MobileNetV3 random init baseline: random weights, ~50%")
    _acc, _t = train_kd(
        student      = _student,
        teacher      = teacher_vgg,
        train_loader = train_loader,
        val_loader   = val_loader,
        device       = device,
        epochs       = KD_EPOCHS,
        lr           = KD_LR_SCR,
        weight_decay = KD_WD,
        temperature  = KD_TEMPERATURE,
        alpha        = KD_ALPHA,
        patience     = KD_PATIENCE,
        save_path    = _ckpt,
    )
    register("vgg_mv3_scratch", _acc)

▶️  Running vgg_mv3_scratch — no checkpoint found
MobileNetV3 random init baseline: random weights, ~50%
Initial student accuracy: 50.00%
Epoch   1/60 | Train 61.06% | Val 64.93% ✅
Epoch   2/60 | Train 65.99% | Val 66.47% ✅
Epoch   3/60 | Train 68.83% | Val 69.40% ✅
Epoch   4/60 | Train 70.44% | Val 69.07%
Epoch   5/60 | Train 72.17% | Val 72.00% ✅
Epoch   6/60 | Train 72.63% | Val 72.40% ✅
Epoch   7/60 | Train 74.03% | Val 72.93% ✅
Epoch   8/60 | Train 73.84% | Val 71.93%
Epoch   9/60 | Train 74.63% | Val 73.20% ✅
Epoch  10/60 | Train 75.39% | Val 75.87% ✅
Epoch  11/60 | Train 75.76% | Val 73.20%
Epoch  12/60 | Train 75.54% | Val 74.67%
Epoch  13/60 | Train 76.97% | Val 76.00% ✅
Epoch  14/60 | Train 77.21% | Val 73.73%
Epoch  15/60 | Train 77.31% | Val 76.20% ✅
Epoch  16/60 | Train 77.84% | Val 76.47% ✅
Epoch  17/60 | Train 77.90% | Val 76.73% ✅
Epoch  18/60 | Train 78.54% | Val 77.07% ✅
Epoch  19/60 | Train 78.37% | Val 77.53% ✅
Epoch  20/60 | Train 78.60% | Val 76.40%
Epoch  21/60 |

---
## Teacher: ResNet_Pretrained  ·  Student: MobileNetV2


In [11]:
# ── ResNet_Pretrained → MobileNetV2 · KD Fine-Tune ────────────────────────
_ckpt = f"{SAVE_DIR}/mv2_kd_resnet_ft.pth"
_run, _reason, _existing_acc = should_run("resnet_mv2_ft", _ckpt, VWW_MobileNetV2)

if not _run:
    print(f"⏭️  Skipping resnet_mv2_ft — {_reason}")
    register("resnet_mv2_ft", _existing_acc, source="existing checkpoint")
else:
    print(f"▶️  Running resnet_mv2_ft — {_reason}")
    _student = VWW_MobileNetV2().to(device)
    _student.load_state_dict(torch.load(MV2_CKPT, map_location=device))
    print(f"MobileNetV2 warm-start baseline: {baseline_mv2*100:.2f}%")
    _acc, _t = train_kd(
        student      = _student,
        teacher      = teacher_resnet,
        train_loader = train_loader,
        val_loader   = val_loader,
        device       = device,
        epochs       = KD_EPOCHS,
        lr           = KD_LR_FT,
        weight_decay = KD_WD,
        temperature  = KD_TEMPERATURE,
        alpha        = KD_ALPHA,
        patience     = KD_PATIENCE,
        save_path    = _ckpt,
    )
    register("resnet_mv2_ft", _acc)

▶️  Running resnet_mv2_ft — no checkpoint found
MobileNetV2 warm-start baseline: 78.40%
Initial student accuracy: 78.40%
Epoch   1/60 | Train 81.31% | Val 77.60%
Epoch   2/60 | Train 81.04% | Val 78.33%
Epoch   3/60 | Train 81.59% | Val 78.27%
Epoch   4/60 | Train 80.81% | Val 78.00%
Epoch   5/60 | Train 82.01% | Val 78.13%
Epoch   6/60 | Train 81.43% | Val 77.33%
Epoch   7/60 | Train 81.83% | Val 76.67%
Epoch   8/60 | Train 81.77% | Val 77.00%
Epoch   9/60 | Train 81.76% | Val 78.20%
Epoch  10/60 | Train 81.94% | Val 77.47%
Epoch  11/60 | Train 82.46% | Val 77.87%
Epoch  12/60 | Train 82.83% | Val 77.27%
Epoch  13/60 | Train 82.40% | Val 79.33% ✅
Epoch  14/60 | Train 83.16% | Val 77.60%
Epoch  15/60 | Train 82.70% | Val 77.93%
Epoch  16/60 | Train 82.91% | Val 78.60%
Epoch  17/60 | Train 82.56% | Val 77.73%
Epoch  18/60 | Train 82.94% | Val 78.00%
Epoch  19/60 | Train 83.01% | Val 78.73%
Epoch  20/60 | Train 83.26% | Val 78.60%
Epoch  21/60 | Train 83.93% | Val 77.33%
Epoch  22/60 | T

In [12]:
# ── ResNet_Pretrained → MobileNetV2 · KD Scratch ──────────────────────────
_ckpt = f"{SAVE_DIR}/mv2_kd_resnet_scratch.pth"
_run, _reason, _existing_acc = should_run("resnet_mv2_scratch", _ckpt, VWW_MobileNetV2)

if not _run:
    print(f"⏭️  Skipping resnet_mv2_scratch — {_reason}")
    register("resnet_mv2_scratch", _existing_acc, source="existing checkpoint")
else:
    print(f"▶️  Running resnet_mv2_scratch — {_reason}")
    _student = VWW_MobileNetV2().to(device)
    print(f"MobileNetV2 random init baseline: random weights, ~50%")
    _acc, _t = train_kd(
        student      = _student,
        teacher      = teacher_resnet,
        train_loader = train_loader,
        val_loader   = val_loader,
        device       = device,
        epochs       = KD_EPOCHS,
        lr           = KD_LR_SCR,
        weight_decay = KD_WD,
        temperature  = KD_TEMPERATURE,
        alpha        = KD_ALPHA,
        patience     = KD_PATIENCE,
        save_path    = _ckpt,
    )
    register("resnet_mv2_scratch", _acc)

▶️  Running resnet_mv2_scratch — no checkpoint found
MobileNetV2 random init baseline: random weights, ~50%
Initial student accuracy: 50.00%
Epoch   1/60 | Train 60.23% | Val 63.73% ✅
Epoch   2/60 | Train 65.09% | Val 63.07%
Epoch   3/60 | Train 67.96% | Val 65.07% ✅
Epoch   4/60 | Train 69.06% | Val 63.13%
Epoch   5/60 | Train 69.84% | Val 71.67% ✅
Epoch   6/60 | Train 71.64% | Val 73.27% ✅
Epoch   7/60 | Train 71.87% | Val 72.33%
Epoch   8/60 | Train 72.20% | Val 72.20%
Epoch   9/60 | Train 73.13% | Val 73.20%
Epoch  10/60 | Train 73.69% | Val 72.87%
Epoch  11/60 | Train 74.54% | Val 74.47% ✅
Epoch  12/60 | Train 74.51% | Val 74.07%
Epoch  13/60 | Train 74.59% | Val 75.80% ✅
Epoch  14/60 | Train 75.90% | Val 75.13%
Epoch  15/60 | Train 75.57% | Val 72.80%
Epoch  16/60 | Train 76.51% | Val 74.60%
Epoch  17/60 | Train 76.94% | Val 74.40%
Epoch  18/60 | Train 76.24% | Val 75.13%
Epoch  19/60 | Train 77.00% | Val 76.47% ✅
Epoch  20/60 | Train 77.96% | Val 76.60% ✅
Epoch  21/60 | Train 77

---
## Teacher: ResNet_Pretrained  ·  Student: MobileNetV3


In [13]:
# ── ResNet_Pretrained → MobileNetV3 · KD Fine-Tune ────────────────────────
_ckpt = f"{SAVE_DIR}/mv3_kd_resnet_ft.pth"
_run, _reason, _existing_acc = should_run("resnet_mv3_ft", _ckpt, VWW_MobileNetV3)

if not _run:
    print(f"⏭️  Skipping resnet_mv3_ft — {_reason}")
    register("resnet_mv3_ft", _existing_acc, source="existing checkpoint")
else:
    print(f"▶️  Running resnet_mv3_ft — {_reason}")
    _student = VWW_MobileNetV3().to(device)
    _student.load_state_dict(torch.load(MV3_CKPT, map_location=device))
    print(f"MobileNetV3 warm-start baseline: {baseline_mv3*100:.2f}%")
    _acc, _t = train_kd(
        student      = _student,
        teacher      = teacher_resnet,
        train_loader = train_loader,
        val_loader   = val_loader,
        device       = device,
        epochs       = KD_EPOCHS,
        lr           = KD_LR_FT,
        weight_decay = KD_WD,
        temperature  = KD_TEMPERATURE,
        alpha        = KD_ALPHA,
        patience     = KD_PATIENCE,
        save_path    = _ckpt,
    )
    register("resnet_mv3_ft", _acc)

▶️  Running resnet_mv3_ft — no checkpoint found
MobileNetV3 warm-start baseline: 79.13%
Initial student accuracy: 79.13%
Epoch   1/60 | Train 82.04% | Val 77.47%
Epoch   2/60 | Train 81.79% | Val 78.00%
Epoch   3/60 | Train 82.24% | Val 78.13%
Epoch   4/60 | Train 82.37% | Val 77.33%
Epoch   5/60 | Train 82.19% | Val 76.40%
Epoch   6/60 | Train 83.23% | Val 78.13%
Epoch   7/60 | Train 81.81% | Val 77.87%
Epoch   8/60 | Train 82.33% | Val 76.47%
Epoch   9/60 | Train 82.69% | Val 77.73%
Epoch  10/60 | Train 82.30% | Val 77.80%
Epoch  11/60 | Train 82.97% | Val 76.80%
Epoch  12/60 | Train 83.30% | Val 77.07%
Epoch  13/60 | Train 82.69% | Val 77.53%
Epoch  14/60 | Train 82.93% | Val 76.80%
Epoch  15/60 | Train 83.77% | Val 77.60%
🛑 Early stopping at epoch 15

✅ Best KD val acc: 79.13%  (3.7 min)
  → Registered [resnet_mv3_ft]: 79.13%  (trained)


In [14]:
# ── ResNet_Pretrained → MobileNetV3 · KD Scratch ──────────────────────────
_ckpt = f"{SAVE_DIR}/mv3_kd_resnet_scratch.pth"
_run, _reason, _existing_acc = should_run("resnet_mv3_scratch", _ckpt, VWW_MobileNetV3)

if not _run:
    print(f"⏭️  Skipping resnet_mv3_scratch — {_reason}")
    register("resnet_mv3_scratch", _existing_acc, source="existing checkpoint")
else:
    print(f"▶️  Running resnet_mv3_scratch — {_reason}")
    _student = VWW_MobileNetV3().to(device)
    print(f"MobileNetV3 random init baseline: random weights, ~50%")
    _acc, _t = train_kd(
        student      = _student,
        teacher      = teacher_resnet,
        train_loader = train_loader,
        val_loader   = val_loader,
        device       = device,
        epochs       = KD_EPOCHS,
        lr           = KD_LR_SCR,
        weight_decay = KD_WD,
        temperature  = KD_TEMPERATURE,
        alpha        = KD_ALPHA,
        patience     = KD_PATIENCE,
        save_path    = _ckpt,
    )
    register("resnet_mv3_scratch", _acc)

▶️  Running resnet_mv3_scratch — no checkpoint found
MobileNetV3 random init baseline: random weights, ~50%
Initial student accuracy: 47.80%
Epoch   1/60 | Train 60.11% | Val 64.20% ✅
Epoch   2/60 | Train 65.46% | Val 65.53% ✅
Epoch   3/60 | Train 67.96% | Val 66.33% ✅
Epoch   4/60 | Train 69.54% | Val 67.27% ✅
Epoch   5/60 | Train 70.83% | Val 71.73% ✅
Epoch   6/60 | Train 71.11% | Val 71.87% ✅
Epoch   7/60 | Train 72.79% | Val 69.73%
Epoch   8/60 | Train 72.50% | Val 73.00% ✅
Epoch   9/60 | Train 73.46% | Val 72.80%
Epoch  10/60 | Train 74.59% | Val 72.93%
Epoch  11/60 | Train 74.51% | Val 74.60% ✅
Epoch  12/60 | Train 75.41% | Val 73.00%
Epoch  13/60 | Train 76.46% | Val 74.80% ✅
Epoch  14/60 | Train 76.74% | Val 74.67%
Epoch  15/60 | Train 76.89% | Val 75.20% ✅
Epoch  16/60 | Train 76.97% | Val 74.00%
Epoch  17/60 | Train 76.60% | Val 74.93%
Epoch  18/60 | Train 77.44% | Val 75.80% ✅
Epoch  19/60 | Train 77.23% | Val 76.53% ✅
Epoch  20/60 | Train 78.31% | Val 75.93%
Epoch  21/60 | 

---
## Results Summary

In [15]:
# ── Reload all best checkpoints and build final table ───────────────
def reload_acc(model_cls, ckpt_name):
    path = f"{SAVE_DIR}/{ckpt_name}"
    if not os.path.exists(path):
        return None
    m = model_cls().to(device)
    m.load_state_dict(torch.load(path, map_location=device))
    return evaluate(m, val_loader, device)

final = {
    "vgg_mv2_ft"       : reload_acc(VWW_MobileNetV2, "mv2_kd_vgg_ft.pth"),
    "vgg_mv2_scratch"  : reload_acc(VWW_MobileNetV2, "mv2_kd_vgg_scratch.pth"),
    "vgg_mv3_ft"       : reload_acc(VWW_MobileNetV3, "mv3_kd_vgg_ft.pth"),
    "vgg_mv3_scratch"  : reload_acc(VWW_MobileNetV3, "mv3_kd_vgg_scratch.pth"),
    "resnet_mv2_ft"    : reload_acc(VWW_MobileNetV2, "mv2_kd_resnet_ft.pth"),
    "resnet_mv2_scratch": reload_acc(VWW_MobileNetV2, "mv2_kd_resnet_scratch.pth"),
    "resnet_mv3_ft"    : reload_acc(VWW_MobileNetV3, "mv3_kd_resnet_ft.pth"),
    "resnet_mv3_scratch": reload_acc(VWW_MobileNetV3, "mv3_kd_resnet_scratch.pth"),
}

def fmt(acc, baseline):
    if acc is None: return f"{'—':>9}   {'—':>7}"
    delta = acc - baseline
    sign  = '+' if delta >= 0 else ''
    return f"{acc*100:>8.2f}%  ({sign}{delta*100:.2f}%)"

W = 72
print("=" * W)
print(f"{'Run':<35} {'Val Acc':>9}  {'Δ baseline':>10}")
print("-" * W)
print(f"{'Teacher VGG_Pretrained':<35} {vgg_acc*100:>8.2f}%")
print(f"{'Teacher ResNet_Pretrained':<35} {resnet_acc*100:>8.2f}%")
print("-" * W)
print(f"{'MobileNetV2 baseline':<35} {baseline_mv2*100:>8.2f}%")
print(f"  VGG    → MobileNetV2  FT      {fmt(final['vgg_mv2_ft'],    baseline_mv2)}")
print(f"  VGG    → MobileNetV2  Scratch  {fmt(final['vgg_mv2_scratch'], baseline_mv2)}")
print(f"  ResNet → MobileNetV2  FT      {fmt(final['resnet_mv2_ft'],    baseline_mv2)}")
print(f"  ResNet → MobileNetV2  Scratch  {fmt(final['resnet_mv2_scratch'], baseline_mv2)}")
print("-" * W)
print(f"{'MobileNetV3 baseline':<35} {baseline_mv3*100:>8.2f}%")
print(f"  VGG    → MobileNetV3  FT      {fmt(final['vgg_mv3_ft'],    baseline_mv3)}")
print(f"  VGG    → MobileNetV3  Scratch  {fmt(final['vgg_mv3_scratch'], baseline_mv3)}")
print(f"  ResNet → MobileNetV3  FT      {fmt(final['resnet_mv3_ft'],    baseline_mv3)}")
print(f"  ResNet → MobileNetV3  Scratch  {fmt(final['resnet_mv3_scratch'], baseline_mv3)}")
print("=" * W)

Run                                   Val Acc  Δ baseline
------------------------------------------------------------------------
Teacher VGG_Pretrained                 89.07%
Teacher ResNet_Pretrained              87.07%
------------------------------------------------------------------------
MobileNetV2 baseline                   78.40%
  VGG    → MobileNetV2  FT         80.00%  (+1.60%)
  VGG    → MobileNetV2  Scratch     79.13%  (+0.73%)
  ResNet → MobileNetV2  FT         79.33%  (+0.93%)
  ResNet → MobileNetV2  Scratch     78.93%  (+0.53%)
------------------------------------------------------------------------
MobileNetV3 baseline                   79.13%
  VGG    → MobileNetV3  FT         79.13%  (+0.00%)
  VGG    → MobileNetV3  Scratch     80.33%  (+1.20%)
  ResNet → MobileNetV3  FT         79.13%  (+0.00%)
  ResNet → MobileNetV3  Scratch     78.93%  (-0.20%)
